# Audio Emotion Recognition — Experiments

Step-by-step training for all audio models × feature sets.
All models use **Bayesian hyperparameter search (optuna)** optimising **dev weighted F1**.

| Model | Feature sets | Search space | Trials |
|---|---|---|---|
| **LR** | egemaps, is10, emobase | C `[1e-3, 10]`, solver, max_iter | 30 |
| **SVM** | egemaps, is10, emobase | C `[1e-3, 10]`, tol, max_iter — `cv=prefit` on dev | 20 |
| **MLP** | egemaps, is10, emobase | hidden_1/2, dropout, lr, focal_gamma | 20 |

Run cells top-to-bottom. Each training cell is independent — re-run any single cell to retrain that model.

## Setup

In [1]:
import sys, json, importlib
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SRC = Path("../src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config import METRICS_DIR, FEATURE_SETS

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

def load_metrics(pattern):
    """Load metrics JSONs. Use {} as placeholder for feature set key."""
    rows = []
    for key in FEATURE_SETS:
        p = METRICS_DIR / pattern.format(key)
        if p.exists():
            m = json.loads(p.read_text())
            rows.append({"feature_set": key,
                         "weighted_f1": m["weighted_f1"],
                         "macro_f1":    m["macro_f1"],
                         "accuracy":    m["accuracy"]})
    return pd.DataFrame(rows).set_index("feature_set") if rows else pd.DataFrame()

print("Setup done. Feature sets:", list(FEATURE_SETS))

Setup done. Feature sets: ['egemaps', 'is10', 'emobase']


---
## 1  Logistic Regression

Bayesian search (30 trials) over `C ∈ [1e-3, 10]`, `solver ∈ {lbfgs, saga}`, `max_iter ∈ [500, 3000]`.  
Objective: **dev weighted F1**.  
Best hyperparameters saved to `outputs/checkpoints/audio_lr_{feature_set}_hparams.json`.

In [ ]:
import audio.lr.train as lr_module
importlib.reload(lr_module)
lr_module.main("egemaps")

Feature set : egemaps (eGeMAPSv02)
Loading features...
Scaler saved.
Bayesian search (30 trials, dev macro-F1)...


  0%|          | 0/30 [00:00<?, ?it/s]

/Users/priyanthvijayasures/Documents/000_Schule/Bachelor Data Science/6. Semester/NLP_Projekte/NLP_P3/.venv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
importlib.reload(lr_module)
lr_module.main("is10")

In [ ]:
importlib.reload(lr_module)
lr_module.main("emobase")

### LR — Results

In [ ]:
lr_results = load_metrics("audio_lr_{}.json")
display(lr_results.style.highlight_max(axis=0, color="#d4f1c4").format("{:.4f}"))

---
## 2  SVM (LinearSVC + Platt calibration)

Bayesian search (20 trials) over `C`, `tol`, `max_iter`.  
`cv='prefit'`: trained on full train set, calibrated on dev — no calibration folds.  
Objective: **dev weighted F1**.

In [ ]:
import audio.svm.train as svm_module
importlib.reload(svm_module)
svm_module.main("egemaps")

In [ ]:
importlib.reload(svm_module)
svm_module.main("is10")

In [ ]:
importlib.reload(svm_module)
svm_module.main("emobase")

### SVM — Results

In [ ]:
svm_results = load_metrics("audio_svm_{}.json")
display(svm_results.style.highlight_max(axis=0, color="#d4f1c4").format("{:.4f}"))

---
## 3  MLP

Bayesian search (20 trials) over architecture `[hidden_1, hidden_2]`, `dropout`, `lr`, `focal_gamma`.  
FocalLoss with balanced class weights. Early stopping patience=10 on **dev weighted F1**.  
Final training: max 100 epochs with ReduceLROnPlateau (factor=0.5, patience=5).  
Outputs to `outputs/checkpoints/` (scaler, best checkpoint, hparams) — one set per feature set.

In [ ]:
import audio.mlp.train as mlp_module
importlib.reload(mlp_module)
mlp_module.main("egemaps")

In [ ]:
importlib.reload(mlp_module)
mlp_module.main("is10")

In [ ]:
importlib.reload(mlp_module)
mlp_module.main("emobase")

### MLP — Results

In [ ]:
mlp_results = load_metrics("audio_mlp_{}.json")
display(mlp_results.style.highlight_max(axis=0, color="#d4f1c4").format("{:.4f}"))

---
## 4  Overall Comparison

In [ ]:
rows = []

for model_tag, pattern in [("LR", "audio_lr_{}.json"), ("SVM", "audio_svm_{}.json"), ("MLP", "audio_mlp_{}.json")]:
    for key in FEATURE_SETS:
        p = METRICS_DIR / pattern.format(key)
        if p.exists():
            m = json.loads(p.read_text())
            rows.append({"model": model_tag, "features": key,
                         "weighted_f1": m["weighted_f1"],
                         "macro_f1":    m["macro_f1"],
                         "accuracy":    m["accuracy"]})

all_results = pd.DataFrame(rows)
display(
    all_results.sort_values("weighted_f1", ascending=False)
               .reset_index(drop=True)
               .style
               .highlight_max(subset=["weighted_f1", "macro_f1"], color="#d4f1c4")
               .format({"weighted_f1": "{:.4f}", "macro_f1": "{:.4f}", "accuracy": "{:.4f}"})
)

In [ ]:
colors = {"LR": "#4C72B0", "SVM": "#DD8452", "MLP": "#55A868"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metric, title in zip(
    axes,
    ["weighted_f1", "macro_f1"],
    ["Weighted F1 — optimisation target", "Macro F1"],
):
    seen_labels = set()
    for _, row in all_results.iterrows():
        label = row["model"] if row["model"] not in seen_labels else None
        ax.bar(
            f"{row['model']}\n{row['features']}",
            row[metric],
            color=colors[row["model"]],
            label=label,
        )
        seen_labels.add(row["model"])
    ax.set_title(title)
    ax.set_ylabel("Score")
    ax.set_ylim(0, 0.65)
    ax.tick_params(axis="x", rotation=30)
    ax.legend()

plt.tight_layout()
plt.savefig("../outputs/plots/audio_model_comparison.png", bbox_inches="tight")
plt.show()

### Per-class breakdown — best model by weighted F1

In [ ]:
best_row  = all_results.loc[all_results["weighted_f1"].idxmax()]
model_key = best_row["model"].lower()
fs_key    = best_row["features"]

best_tag  = f"audio_{model_key}_{fs_key}"

print(f"Best model : {best_tag}")
print(f"Weighted F1: {best_row['weighted_f1']:.4f}  |  Macro F1: {best_row['macro_f1']:.4f}")

m = json.loads((METRICS_DIR / f"{best_tag}.json").read_text())
per_class = pd.DataFrame(m["per_class"]).T[["precision", "recall", "f1", "support"]]
display(
    per_class.style
             .background_gradient(subset=["f1"], cmap="RdYlGn", vmin=0, vmax=0.7)
             .format({"precision": "{:.3f}", "recall": "{:.3f}",
                      "f1": "{:.3f}", "support": "{:.0f}"})
)